In [1]:
from calendar import day_name

import pandas as pd
import numpy as np
from datetime import timedelta


In [2]:
customers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_items_dataset.csv')
order_payment = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_payments_dataset.csv')
order_review = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_sellers_dataset.csv')
products_category_name_translation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/product_category_name_translation.csv')


In [3]:
cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']

for col in cols:
    orders[col] = pd.to_datetime(orders[col],errors='coerce')

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [5]:
delivery_time = orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
# delivery_time.head()


In [6]:
negative_delivery_time = (delivery_time < pd.Timedelta(0)).sum()

print("Negative delivery times:", negative_delivery_time)


Negative delivery times: 0


In [7]:
delivery_time_seconds = delivery_time.dt.total_seconds()
orders['delivery_time_days'] = delivery_time_seconds / 86400
# delivery_time_days

In [8]:
orders.delivery_time_days.describe()

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

In [9]:
orders.delivery_time_days[orders.delivery_time_days > 15.720327].count()

np.int64(24119)

In [10]:
outliers = orders.delivery_time_days[orders.delivery_time_days > 29.1512]
outliers.describe()

count    4899.000000
mean       40.783683
std        16.205274
min        29.153322
25%        32.142616
50%        36.153600
75%        43.724716
max       209.628611
Name: delivery_time_days, dtype: float64

In [11]:
orders.delivery_time_days.sort_values(ascending=False).head(10)

19590    209.628611
55619    208.351759
61610    195.634016
70307    194.850174
89130    194.633611
38509    194.049583
11399    191.463542
81401    189.863160
54480    188.134618
68769    187.743843
Name: delivery_time_days, dtype: float64

In [12]:
orders.sort_values(by='delivery_time_days', ascending=False).head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days
19590,ca07593549f1816d26a572e06dc1eab6,75683a92331068e2d281b11a7866ba44,delivered,2017-02-21 23:31:27,2017-02-23 02:35:15,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22,209.628611
55619,1b3190b2dfa9d789e1f14c05b647a14a,d306426abe5fca15e54b645e4462dc7b,delivered,2018-02-23 14:57:35,2018-02-23 15:16:14,2018-02-26 18:49:07,2018-09-19 23:24:07,2018-03-15,208.351759
61610,440d0d17af552815d15a9e41abe49359,7815125148cfa1e8c7fee1ff7974f16c,delivered,2017-03-07 23:59:51,2017-03-09 01:11:33,2017-03-15 13:00:08,2017-09-19 15:12:50,2017-04-07,195.634016
70307,2fb597c2f772eca01b1f5c561bf6cc7b,217906bc11a32c1e470eb7e08584894b,delivered,2017-03-08 18:09:02,2017-03-08 18:09:02,2017-03-13 08:54:02,2017-09-19 14:33:17,2017-04-17,194.850174
89130,285ab9426d6982034523a855f55a885e,9cf2c3fa2632cee748e1a59ca9d09b21,delivered,2017-03-08 22:47:40,2017-03-08 22:47:40,2017-03-09 08:53:20,2017-09-19 14:00:04,2017-04-06,194.633611
38509,0f4519c5f1c541ddec9f21b3bddd533a,1a8a4a30dc296976717f44e7801fdeef,delivered,2017-03-09 13:26:57,2017-03-09 13:26:57,2017-03-22 05:28:14,2017-09-19 14:38:21,2017-04-11,194.049583
11399,47b40429ed8cce3aee9199792275433f,cb2caaaead400c97350c37a3fc536867,delivered,2018-01-03 09:44:01,2018-01-03 10:31:15,2018-02-06 01:48:28,2018-07-13 20:51:31,2018-01-19,191.463542
81401,2fe324febf907e3ea3f2aa9650869fa5,65b14237885b3972ebec28c0f7dd2220,delivered,2017-03-13 20:17:10,2017-03-13 20:17:10,2017-03-17 07:23:53,2017-09-19 17:00:07,2017-04-05,189.863160
54480,2d7561026d542c8dbd8f0daeadf67a43,8199345f57c6d1cbe9701f92481beb8d,delivered,2017-03-15 11:24:27,2017-03-15 11:24:27,2017-03-16 09:29:16,2017-09-19 14:38:18,2017-04-13,188.134618
68769,c27815f7e3dd0b926b58552628481575,f85e9ec0719b16dc4dd0edd438793553,delivered,2017-03-15 23:23:17,2017-03-15 23:23:17,2017-03-21 14:50:25,2017-09-19 17:14:25,2017-04-10,187.743843


In [13]:
orders[orders['order_delivered_customer_date'].dt.date == pd.to_datetime('2017-09-19').date()].count()

order_id                         282
customer_id                      282
order_status                     282
order_purchase_timestamp         282
order_approved_at                282
order_delivered_carrier_date     282
order_delivered_customer_date    282
order_estimated_delivery_date    282
delivery_time_days               282
dtype: int64

In [14]:
orders['order_delivered_date'] = orders['order_delivered_customer_date'].dt.date
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,order_delivered_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,2017-10-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,2018-08-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,2018-08-17
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,2017-12-02
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,2018-02-16


In [15]:
orders.groupby('order_delivered_date').size().max()

np.int64(446)

In [16]:
orders.groupby('order_delivered_date').size().sort_values(ascending=False).head(10)

order_delivered_date
2018-08-27    446
2018-08-13    442
2018-05-14    434
2018-05-21    431
2018-05-18    425
2018-04-11    413
2017-12-11    412
2018-07-03    410
2018-05-03    409
2018-07-30    405
dtype: int64

In [17]:
temp_df = orders[orders['delivery_time_days'] > 29.1512]
temp_df.groupby('order_delivered_date').size().sort_values(ascending=False).head(20)

order_delivered_date
2018-04-03    67
2018-01-04    66
2018-04-10    63
2018-01-09    63
2018-04-11    63
2018-01-08    62
2018-04-16    60
2017-12-27    60
2018-04-04    58
2017-09-19    57
2018-01-03    56
2018-04-18    55
2018-04-17    53
2017-12-28    50
2018-01-05    50
2018-04-05    48
2018-04-19    47
2018-04-02    45
2018-04-20    45
2018-04-13    44
dtype: int64

In [18]:
diff = temp_df['order_estimated_delivery_date'] - temp_df['order_delivered_customer_date']
diff.sort_values(ascending=False)

14578      55 days 05:24:30
92724      51 days 10:17:07
59011      40 days 11:35:35
78955      40 days 02:20:27
37224      37 days 06:35:33
                ...        
89130   -167 days +09:59:56
81401   -168 days +06:59:53
11399   -176 days +03:08:29
19590   -182 days +09:23:21
55619   -189 days +00:35:53
Length: 4899, dtype: timedelta64[us]

In [19]:
late = diff < pd.Timedelta(0)
late.sum()


np.int64(3915)

In [20]:
median_lateness = -(diff[late].dt.total_seconds() / 86400).median()
median_lateness

np.float64(11.679097222222222)

In [21]:
delivery_diff = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date'])

late_orders = delivery_diff > pd.Timedelta(0)

late_count = late_orders.sum()

valid_orders = delivery_diff.notna().sum()

late_rate = (late_count / valid_orders) * 100

late_count, valid_orders, late_rate

(np.int64(7827), np.int64(96476), np.float64(8.112898544715783))

In [22]:
delivery_diff[late_orders].describe()

count                       7827
mean      9 days 13:15:10.612878
std      13 days 22:50:26.879558
min              0 days 00:03:36
25%       1 days 20:48:45.500000
50%              5 days 19:23:22
75%      11 days 19:44:21.500000
max            188 days 23:24:07
dtype: object

In [29]:
sellers = orders['order_delivered_carrier_date'] - orders['order_purchase_timestamp']
sellers.median()

Timedelta('2 days 04:54:44')

In [34]:
op = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
op.mean()
op.median()

Timedelta('7 days 02:23:40')

In [35]:
freight_by_order = order_items.groupby('order_id')['freight_value'].sum()

delivery_freight = orders[['order_id', 'delivery_time_days']].copy()

delivery_freight['freight_value'] = delivery_freight['order_id'].map(freight_by_order)

delivery_freight[['delivery_time_days', 'freight_value']].describe()

,delivery_time_days,freight_value
count,96476.000000,98666.000000
mean,12.558702,22.823562
std,9.546530,21.650909
min,0.533414,0.000000
25%,6.766403,13.850000
50%,10.217755,17.170000
75%,15.720327,24.040000
max,209.628611,1794.960000
